In [1]:
import pandas as pd
import numpy as np

In [4]:
input_size = 3
hidden_size = 128
num_layers = 3
output_size = 1
dropout = 0.2

model = RNN_DAP(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, output_size=output_size, dropout=dropout)

In [5]:
model.load_state_dict(torch.load("/Users/florian/Documents/github/thesis/stochastic-optimization/EnergyStorage_II/Notebooks/rnn_model.pth", weights_only=True))
scaler = joblib.load("/Users/florian/Documents/github/thesis/stochastic-optimization/EnergyStorage_II/Notebooks/scaler.pkl")

In [ ]:
data = pd.DataFrame([55, 51, 56])

data_prep = data.values.reshape(1, -1)
data_prep.shape

In [ ]:
data.shape

In [8]:
data_scaled = scaler.transform(data_prep)

In [9]:
data_scaled_tensor = torch.tensor(data_scaled, dtype=torch.float32).view(1, 1, 3)

In [10]:
y_pred = model(data_scaled_tensor)

In [ ]:
y_pred.item()

In [ ]:
test = pd.DataFrame([55, 50, 60]).values
test

In [13]:
test = pd.DataFrame([[55, 57, 60]]).values
test_scaled = scaler.transform(test.reshape(1, -1))
test_tensor = torch.tensor(test_scaled, dtype=torch.float32).view(1, 1, 3)
steps = 3

for _ in range(steps):
    y_pred = model(test_tensor).item()
    y_pred_scaled = scaler.transform(np.array([[y_pred, y_pred, y_pred]]).reshape(1, -1)).flatten()[-1]
    test_tensor = torch.cat((test_tensor[:, :, 1:], torch.tensor([[[y_pred_scaled]]], dtype=torch.float32)), dim=2)

test_final = scaler.inverse_transform(test_tensor.numpy().reshape(1, -1))

In [ ]:
test = pd.DataFrame([[55, 57, 60]])
test.values.reshape(1, -1)

In [ ]:
test = pd.DataFrame([55, 57, 60])
test.values.reshape(1, -1)

In [ ]:
import pandas as pd
import numpy as np
import torch

test = pd.DataFrame([[55, 57, 60]]).values
print(type(test))
test_scaled = scaler.transform(test.reshape(1, -1))
test_tensor = torch.tensor(test_scaled, dtype=torch.float32).view(1, 1, 3)
steps = 3

for i in range(steps):
    y_pred = model(test_tensor).item()
    y_pred_scaled = scaler.transform(np.array([[y_pred, y_pred, y_pred]]).reshape(1, -1)).flatten()[-1]
    if i < test_tensor.size(2):
        test_tensor = torch.cat((torch.tensor([[[y_pred_scaled]]], dtype=torch.float32), test_tensor[:, :, :-1]), dim=2)
    else:
        test_tensor = torch.cat((test_tensor[:, :, :3], torch.tensor([[[y_pred_scaled]]], dtype=torch.float32)), dim=2)

test_final = scaler.inverse_transform(test_tensor.numpy().resh
                                      ape(1, -1)).flatten()

print(test_final)

In [92]:
def make_rnn_forecast(data: np.ndarray=None, steps: int=3):
    data_scaled = scaler.transform(data.reshape(1, -1))
    data_tensor = torch.tensor(data_scaled, dtype=torch.float32).view(1, 1, -1)
    
    model.eval()
    forecast = []
    with torch.no_grad():
        for _ in range(steps):
            y_pred = model(data_tensor).item()
            forecast.append(y_pred)
            # Update data_tensor with the new prediction
            y_pred_scaled = scaler.transform(np.array([[y_pred] * data_tensor.shape[2]])).flatten()[-1]
            data_tensor = torch.cat((torch.tensor([[[y_pred_scaled]]], dtype=torch.float32), data_tensor[:, :, :-1]), dim=2)
    
    forecast = np.array(forecast).reshape(-1, 1)
    # Rückskaliere die Vorhersagen manuell
    mean = scaler.mean_[0]
    print(mean)
    std = scaler.scale_[0]
    print(std)
    forecast = forecast * std + mean

    return forecast.flatten()

In [161]:
def make_rnn_forecast(data: np.ndarray=None, steps: int=3):
        data_scaled = scaler.transform(data.reshape(1, -1))
        data_tensor = torch.tensor(data_scaled, dtype=torch.float32).view(1, 1, -1)
        
        model.eval()
        with torch.no_grad():
            for i in range(steps):
                input_tensor = data_tensor[:, :, :3]
                print(input_tensor)
                y_pred = model(input_tensor).item()
                y_pred_scaled = scaler.transform(np.array([[y_pred, y_pred, y_pred]]).reshape(1, -1)).flatten()[-1]
                if i < data_tensor.size(2):
                    data_tensor = torch.cat((torch.tensor([[[y_pred_scaled]]], dtype=torch.float32), data_tensor[:, :, :-1]), dim=2)
                else:
                    print(1)
                    data_tensor = torch.cat((torch.tensor([[[y_pred_scaled]]], dtype=torch.float32), data_tensor), dim=2)
        forecast = data_tensor.numpy().reshape(-1)
        mean = scaler.mean_[0]
        std = scaler.scale_[0]
        forecast = forecast * std + mean
        
        return forecast

In [ ]:
array = make_rnn_forecast(data=np.array([77, 12, 66, 67, 68]), steps=7)
array

In [ ]:
for i in range(5):
    print(f"price_tp{i+1}")

In [ ]:
for i in range(5):
    print(f"price_tp{i+1}, {i}")

In [ ]:
import torch
import numpy as np
import joblib
import sys
sys.path.append("../")
from StochasticModels.CNFWithAR import CNFWithARModel, Context_LSTM


context_lstm_1 = Context_LSTM(
    input_size=24,
    hidden_size=256,
    num_layers=1,
    output_size=120,
    dropout=0.0
)

cnf_armodel = CNFWithARModel(
    x_dim=1,
    c_dim=120,
    num_flows=2,
    weight_decay=1e-5,
    temperature=1.5,
    dropout_probability=0.0,
    num_blocks=0,
    hidden_features=64,
    use_batch_norm=True,
    use_residual_blocks=False,
    ar_model=context_lstm_1
)

c_scaler = joblib.load("/Users/florian/Documents/github/thesis/stochastic-optimization/EnergyStorage_II/StochasticModels/models/lstm_maf_context_scaler.pkl")
X_scaler = joblib.load("/Users/florian/Documents/github/thesis/stochastic-optimization/EnergyStorage_II/StochasticModels/models/lstm_maf_X_scaler.pkl")

cnf_armodel.load("/Users/florian/Documents/github/thesis/stochastic-optimization/EnergyStorage_II/StochasticModels/models/lstm_maf_forecast_1.pth")

In [1]:
import torch 
import numpy as np
import joblib
import sys
sys.path.append("../")

from Forecast.ForecastModels import LSTM_MAF_old, MLP_MAF

test = MLP_MAF("MLP_MAF")

In [40]:
import pandas as pd
import torch
import numpy as np
forecast_df = pd.read_csv("../Data/forecast.csv")
policy_df = pd.read_csv("../Data/policy.csv")

def get_1_day_lag(df):
    df_copy = df.copy()
    for lag in range(1, 25):
        df_copy['lag_{}'.format(lag)] = df_copy["DAP DE/LU"].shift(lag)
    df_copy = df_copy.dropna()
    df_copy = df_copy.reset_index(drop=True)    
    return df_copy


train_df = forecast_df[["DAP DE/LU"]].copy()
val_df = policy_df[["DAP DE/LU"]].copy()
train_df = get_1_day_lag(train_df)
val_df = get_1_day_lag(val_df)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Train data
X = train_df[["DAP DE/LU"]]
c = train_df.drop(columns=["DAP DE/LU"])

c_train, c_test, X_train, X_test = train_test_split(c, X, test_size=0.5, shuffle=False)

X_scaler = StandardScaler()
c_scaler = StandardScaler()

X_train_scaled = X_scaler.fit_transform(X_train)
c_train_scaled = c_scaler.fit_transform(c_train)

X_test_scaled = X_scaler.transform(X_test)
c_test_scaled = c_scaler.transform(c_test)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
c_train_tensor = torch.tensor(c_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
c_test_tensor = torch.tensor(c_test_scaled, dtype=torch.float32)


# Validation data
X_val = val_df[["DAP DE/LU"]]
context_val = val_df.drop(columns=["DAP DE/LU"])

c_val_scaled = c_scaler.transform(context_val)
X_val_scaled = X_scaler.transform(X_val)

X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
c_val_tensor = torch.tensor(c_val_scaled, dtype=torch.float32)

In [41]:
data = np.array(context_val[:1])

In [ ]:
context_val[:1].shape

In [43]:
def predict_1(model, c_scaler, X_scaler, data, num_samples_paths, steps):

    if not isinstance(data, np.ndarray):
        raise ValueError("Data must be a numpy array")
    data_scaled = c_scaler.transform(data)
    data_tensor = torch.tensor(data_scaled, dtype=torch.float32).unsqueeze(1)  
    predictions = []
    model.eval()
    with torch.no_grad():
        for i in range(steps):
            y_samples = model.generate_sample_paths(num_sample_paths=num_samples_paths, num_samples=i+1, context=data_tensor)
            y_samples = torch.tensor(y_samples, dtype=torch.float32)
            y_samples_mean = y_samples.mean(dim=0, keepdim=True)
            predictions.append(y_samples_mean[:, i, :].numpy().item())
            last_row = data_tensor[-1, :, :-1]
            new_row = torch.cat((y_samples_mean[:, i, :], last_row), dim=1)
            data_tensor = torch.cat((data_tensor, new_row.unsqueeze(0)), dim=0)

    predictions = np.array(predictions).reshape(-1, 1)
    rescaled_samples = X_scaler.inverse_transform(predictions)
    return rescaled_samples

In [ ]:
forecast = predict_1(model=cnf_armodel, c_scaler=c_scaler, X_scaler=X_scaler, data=data, steps=100*24, num_samples_paths=10)

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()   
fig.add_trace(go.Scatter(x=X_val[:100*24].index, y=X_val[:100*24].values.flatten(), mode='lines', name='True'))
fig.add_trace(go.Scatter(x=X_val[:100*24].index, y=forecast.flatten(), mode="lines", name="Forecast"))

In [47]:
from ForecastModels import LSTM_MAF

In [ ]:
data

In [ ]:
import sys
sys.path.append("../")

model = LSTM_MAF(
    model_name = "lstm-maf"
)

In [ ]:
p = 2400

forecast = model.predict(data=data, steps=p, num_sample_paths=1)

import plotly.graph_objects as go

fig = go.Figure()   
fig.add_trace(go.Scatter(x=X_val[:p].index, y=X_val[:p].values.flatten(), mode='lines', name='True'))
fig.add_trace(go.Scatter(x=X_val[:p].index, y=forecast.flatten(), mode="lines", name="Forecast"))